In [ ]:
import os
import zipfile
from kaggle.api.kaggle_api_extended import KaggleApi

DATASET = "cdc/chronic-disease"
DATA_PATH = os.path.join("datasets", "chronic_disease")

def fetch_chronic_disease_data(dataset=DATASET, data_path=DATA_PATH):
    if not os.path.isdir(data_path):
        os.makedirs(data_path)

    # Authenticate Kaggle
    api = KaggleApi()
    api.authenticate()

    # Download dataset (this downloads a zip file)
    api.dataset_download_files(dataset, path=data_path, unzip=True)

    print("Download complete!")

In [ ]:
fetch_chronic_disease_data()

In [ ]:
def load_chronic_data(data_path=DATA_PATH):
    for file in os.listdir(data_path):
        if file.endswith(".csv"):
            return pd.read_csv(os.path.join(data_path, file))
    raise FileNotFoundError("No CSV file found in directory")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sklearn.linear_model

In [ ]:
chronic = load_chronic_data()

In [ ]:
chronic.head()

In [ ]:
chronic.info()

In [ ]:
chronic.Stratification1.value_counts()

In [ ]:
plt.figure(figsize = (14,6))
sns.countplot(data = chronic, y = 'Topic')
plt.xticks(rotation = 45)
plt.title('Counts of each health topic')
plt.show()


This graph depicts the counts of different health topics discussed in the survey. As seen, some of the most common topics include diabetes, cardiovascular disease, and chronic obstructive pulmonary disease (COPD). This is across all states and territories included in the survey. This also spans all of the years of the survey. Some of the least common topics include disability, reproductive health, immunization, and mental health.

In [ ]:
# Pick one topic and one measure type to keep things clean
diabetes = chronic[chronic['Topic'] == 'Diabetes']
diabetes_overall = diabetes[diabetes['Stratification1'] == 'Overall']
diabetes_overall = diabetes_overall[
    ~diabetes_overall['LocationDesc'].str.contains('United States', na=False)
]
diabetes_overall = diabetes_overall[
    diabetes_overall['DataValueType'] == 'Age-adjusted Prevalence'
]
diabetes_overall = diabetes_overall[
    diabetes_overall['Question'] == 'Prevalence of diagnosed diabetes among adults aged >= 18 years'
]

In [ ]:
# Average diabetes values by state
state_diabetes = (
    diabetes_overall
    .groupby('LocationDesc')['DataValueAlt']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

In [ ]:
print(diabetes_overall['DataValueType'].value_counts())

In [ ]:
print(diabetes_overall['Question'].value_counts())

In [ ]:
# Plot
plt.figure(figsize=(12, 10))
sns.barplot(data=state_diabetes, y='LocationDesc', x='DataValueAlt')

plt.title("Average Diabetes Prevalence Rate by State")
plt.xlabel("Average Prevalence Rate")
plt.ylabel("Location")
plt.show()

Zeroing in on one health topic, Diabetes, we look at its prevalence by location. We had to clean the data significantly and couldn't just use the DataAltValue category as a data prevalence type becuse it mixes prevalence rates with raw counts for examples which could skew more heavily populated states to looking like they have higher rates of diabetes. We also had to clean the questions so they just had to do with diabetes prevalence and not other factors. After filtering, the locations with the highest prevalences include Puerto Rico, Guam Mississippi, and Alabama.

In [ ]:
# Pick one topic and one measure type to keep things clean
alc = chronic[chronic['Topic'] == 'Alcohol']
alc_overall = alc[alc['Stratification1'] == 'Overall']
alc_overall = alc_overall[
    alc_overall['DataValueType'] == 'Age-adjusted Prevalence'
]
alc_overall = alc_overall[
    alc_overall['Question'] == 'Binge drinking prevalence among adults aged >= 18 years'
]

In [ ]:
print(alc_overall['DataValueType'].value_counts())

In [ ]:
print(alc_overall['Question'].value_counts())

In [ ]:
year_alc = (
    alc_overall
    .groupby(['LocationAbbr','YearEnd'])['DataValueAlt']
    .mean()
    .sort_values(ascending=True)
    .reset_index()
)
states = ['NY', 'CA', 'PR', 'WY', 'KY']

In [ ]:
# Plot
plt.figure(figsize=(12, 10))
sns.lineplot(data=year_alc[year_alc['LocationAbbr'].isin(states)], y='DataValueAlt', x='YearEnd', hue = 'LocationAbbr')

plt.title("Average Alcohol Prevalence Data Value 2011-2015")
plt.xlabel("Year of the experiment")
plt.ylabel("Average Prevalence Rate")
plt.show()

In general, many locations followed the trajectory of alcohol prevalence going downwards from 2011 to 2012 and up from 2014 to 2015. Between 2011 and 2014, though, the trajectories are variable. I chose these states so we could track states with low populations and states with high ones to see the differences. I also specifically selected states and territories where alcohol is produced such as Kentucky, Puerto Rico, and California to see if there are higher rates of binge drinking, but there doesn't seem to be much evidence for that.

In [ ]:
# Pick one topic and one measure type to keep things clean
cvd = chronic[chronic['Topic'] == 'Cardiovascular Disease']
cvd_overall = cvd[cvd['Stratification1'].isin(['Female', 'Male'])]
cvd_overall = cvd_overall[
    cvd_overall['DataValueType'] == 'Age-adjusted Rate'
]
cvd_overall = cvd_overall[
    cvd_overall['Question'] == 'Mortality from total cardiovascular diseases'
]


In [ ]:
print(cvd_overall['DataValueType'].value_counts())

In [ ]:
print(cvd_overall['Question'].value_counts())

In [ ]:
genderid_cvd = (
    cvd_overall
    .groupby(['LocationAbbr','Stratification1'])['DataValueAlt']
    .mean()
    .sort_values(ascending=True)
    .reset_index()
)



In [ ]:
# Plot
plt.figure(figsize=(12, 10))
sns.barplot(data=genderid_cvd, x='DataValueAlt', y='LocationAbbr', hue = 'Stratification1')
plt.title("Cardiovascular mortality rate differences between men and women")
plt.ylabel("Location")
plt.xlabel("Age adjusted mortality rate (per 100,000)")
plt.show()

This bar chart shows cardiovascular disease and the differences between men and women. In all locations, men have a higher mortality rate than women. It could be interesting to find which states have the smallest differences and investigate why that might be and if there's any correlation between a small gap between men and women and lower mortality rates overall. It might also be worth investigating the commonalities in locations with the lowest female mortality rates. 

In [ ]:
diabetes = chronic[(chronic['Topic'] == 'Diabetes') & 
    (chronic['LocationDesc'] != 'United States') &
    (chronic['Stratification1'].isin(['Female', 'Male']))]

In [ ]:
year_diabetes = (
    diabetes_overall
    .groupby(['LocationAbbr','YearEnd'])['DataValueAlt']
    .mean()
    .sort_values(ascending=True)
    .reset_index()
)

In [ ]:
plt.figure(figsize=(12, 10))
# sns.scatterplot(data = year_diabetes, y = 'YearEnd', x = 'DataValueAlt')
sns.regplot(y = 'DataValueAlt', x = 'YearEnd', data = diabetes_overall)
plt.xlabel('Year')
plt.ylabel('Diabetes prevalence')
plt.title('Linear regression of Diabetes prevalence from 2011-2015')
plt.show()

Here, we plot linear regression on Diabetes prevalence versus the year that the experiment ended. The plot isn't very informative and I think having more years or the data taken per month could have given a better plot. From my observation, this data is too categorical for insightful regression. 